**Dependency Installation**

In [ ]:
!pip install kaggle --quiet

**Food-101 Direct Download**

In [ ]:
!wget -q http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz
!tar -xzf food-101.tar.gz
!ls food-101/images | head

apple_pie
baby_back_ribs
baklava
beef_carpaccio
beef_tartare
beet_salad
beignets
bibimbap
bread_pudding
breakfast_burrito


**KaggleAPI Token Initialization**

In [ ]:
!mkdir -p ~/.kaggle
!echo KGAT_e2bc4a074b3e3820363df6ec98be59d4 > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

**Raw Dataset Installation**

In [ ]:
!kaggle datasets download -d kritikseth/fruit-and-vegetable-image-recognition
!unzip -q fruit-and-vegetable-image-recognition.zip -d raw_food_dataset
!ls raw_food_dataset

Dataset URL: https://www.kaggle.com/datasets/kritikseth/fruit-and-vegetable-image-recognition
License(s): CC0-1.0
100% 1.98G/1.98G [00:21<00:00, 99.1MB/s]

test  train  validation


**Training Script Upload**

In [ ]:
from google.colab import files
files.upload()

Saving train_combined_classifier.py to train_combined_classifier.py


{'train_combined_classifier.py': b'"""\n===================================================================\nCombined Food Classifier - Prepared Dishes (Food-101) + Raw Produce\n===================================================================\nPurpose:\n    Train a single classifier that recognizes BOTH:\n      - 101 prepared dish categories (Food-101, downloaded directly\n        from the original source - NOT via tensorflow_datasets, to\n        avoid protobuf version conflicts in Colab)\n      - Raw fruits/vegetables (local folder dataset, e.g. Kaggle\'s\n        "Fruits and Vegetables Image Recognition Dataset" by Kritik Seth)\n\n    Output is one combined label space (101 + 36 = 137 classes).\n\nBefore running, download BOTH datasets as plain folders:\n\n    Food-101 (downloads ~5GB, already organized by class):\n        !wget -q http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz\n        !tar -xzf food-101.tar.gz\n    Expected: food-101/images/<dish_name>/*.jpg  (101 folders)\n

In [ ]:
!grep food101_dir train_combined_classifier.py

        --food101_dir ./food-101/images \
    parser.add_argument("--food101_dir", type=str, required=True,
    train_ds_dishes = load_split_from_single_dir(args.food101_dir, "training")
    val_ds_dishes = load_split_from_single_dir(args.food101_dir, "validation")


**Test Run (1 Epoch)**

In [ ]:
!python train_combined_classifier.py --food101_dir ./food-101/images --raw_data_dir ./raw_food_dataset --epochs 1

2026-06-24 03:35:41.518169: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Found 101000 files belonging to 101 classes.
Using 80800 files for training.
2026-06-24 03:35:52.933101: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1782272152.934567    2805 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Found 101000 files belonging to 101 classes.
Using 20200 files for validation.
Found 3115 files belonging to 36 classes.
Found 351 files belonging to 36 classes.
Comb

**Training (20 Epochs)**

In [ ]:
!python train_combined_classifier.py --food101_dir ./food-101/images --raw_data_dir ./raw_food_dataset --epochs 20

2026-06-24 03:48:43.047167: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Found 101000 files belonging to 101 classes.
Using 80800 files for training.
2026-06-24 03:48:51.795628: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1782272931.797094    6312 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Found 101000 files belonging to 101 classes.
Using 20200 files for validation.
Found 3115 files belonging to 36 classes.
Found 351 files belonging to 36 classes.
Comb

**Fine-Tuning AI Accuracy**

In [ ]:
from google.colab import files
files.upload()

Saving finetune_classifier.py to finetune_classifier.py


{'finetune_classifier.py': b'"""\n===================================================================\nFine-Tuning Phase 2 - Unfreeze MobileNetV2 base layers\n===================================================================\nPurpose:\n    Your first training run (20 epochs, head-only) plateaued around\n    58% validation accuracy because MobileNetV2\'s base layers were\n    frozen - only the small classification head was learning.\n\n    This script picks up from that already-trained model, unfreezes\n    the LAST N layers of the base network, and continues training\n    at a much lower learning rate. This lets those layers adapt\n    specifically to food images instead of staying generic ImageNet\n    features - this is the step that typically gives the real\n    accuracy jump in transfer learning.\n\nBefore running:\n    Upload your previous run\'s \'best_model.keras\' (from model_output/)\n    alongside this script, and have food-101/images and raw_food_dataset\n    available exa

In [ ]:
!grep -n "head_layer_count" finetune_classifier.py

136:    head_layer_count = 4
137:    base_layers = model.layers[:-head_layer_count]
138:    head_layers = model.layers[-head_layer_count:]
150:          f"(plus the {head_layer_count} head layers, always trainable).")


In [ ]:
!python finetune_classifier.py --previous_model ./model_output/best_model.keras --food101_dir ./food-101/images --raw_data_dir ./raw_food_dataset --epochs 10 --unfreeze_layers 30

2026-06-24 06:01:45.986868: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Found 101000 files belonging to 101 classes.
Using 80800 files for training.
2026-06-24 06:01:53.413345: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1782280913.414810   41178 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.
Found 101000 files belonging to 101 classes.
Using 20200 files for validation.
Found 3115 file

**Obtain Trained Model**

In [ ]:
from google.colab import files
files.download('model_output/food_classifier.tflite')
files.download('model_output/class_indices.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>